In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import numpy as np
import random
import matplotlib.pyplot as plt
from tqdm import tqdm

# ===============================
# 1. Define Paths
# ===============================
input_root = "/content/drive/MyDrive/defense/Dataset"
output_root = "/content/drive/MyDrive/defense/preprocessed"

# ===============================
# 2. Create Output Root Folder
# ===============================
if not os.path.exists(output_root):
    os.makedirs(output_root)

# ===============================
# 3. Preprocessing Function
# ===============================
def preprocess_image(image_path, size=(224, 224)):
    """
    Resize, noise reduction, normalization
    """
    img = cv2.imread(image_path)

    if img is None:
        print(f"Warning: Could not read {image_path}")
        return None

    # Resize
    img = cv2.resize(img, size)



    # CLAHE
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)

    limg = cv2.merge((cl, a, b))
    img = cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

    # Normalize 0-1
    img = img / 255.0

    # Convert back to uint8 for saving
    img = (img * 255).astype(np.uint8)

    return img

# ===============================
# 4. Loop Through Dataset Folders
# ===============================
for disease_name in os.listdir(input_root):
    disease_input_path = os.path.join(input_root, disease_name)
    disease_output_path = os.path.join(output_root, disease_name)

    # Skip if not a folder
    if not os.path.isdir(disease_input_path):
        continue

    # Create disease folder in output
    if not os.path.exists(disease_output_path):
        os.makedirs(disease_output_path)

    # Loop through files in disease folder
    for file_name in tqdm(os.listdir(disease_input_path), desc=disease_name):
        input_img_path = os.path.join(disease_input_path, file_name)

        # Skip folders
        if os.path.isdir(input_img_path):
            continue

        # Only process image files
        if not file_name.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        # Preprocess
        processed_img = preprocess_image(input_img_path)

        # Save if valid
        if processed_img is not None and processed_img.size != 0:
            # Use .jpg extension for all
            filename, _ = os.path.splitext(file_name)
            output_img_path = os.path.join(disease_output_path, filename + ".jpg")
            cv2.imwrite(output_img_path, processed_img)

In [ ]:
import os
import shutil
import random

# ====== PATH ======
input_dir = "/content/drive/MyDrive/defense/preprocessed"
output_dir = "/content/drive/MyDrive/defense/dataset_split"

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

# Create main folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_dir, split), exist_ok=True)

# ====== PROCESS ======
for disease in os.listdir(input_dir):
    disease_path = os.path.join(input_dir, disease)

    if not os.path.isdir(disease_path):
        continue

    images = os.listdir(disease_path)
    random.shuffle(images)

    total = len(images)

    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))

    train_imgs = images[:train_end]
    val_imgs = images[train_end:val_end]
    test_imgs = images[val_end:]

    # Create disease folders inside each split
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(output_dir, split, disease), exist_ok=True)

    # ====== COPY FILES ======
    def copy_images(img_list, split_name):
        for img in img_list:
            src = os.path.join(disease_path, img)
            dst = os.path.join(output_dir, split_name, disease, img)
            shutil.copy(src, dst)

    copy_images(train_imgs, "train")
    copy_images(val_imgs, "val")
    copy_images(test_imgs, "test")

# ====== DONE ======
print("✅ Dataset split complete!")
